In [ ]:
import numpy as np, pandas as pd, gurobipy as gp
from gurobipy import GRB
from collections import defaultdict

L = np.load("L_metres.npy"); site_dist = np.load("site_dist.npy")       # from notebook 03

sites = pd.read_csv("sites.csv")
d = pd.read_csv("demand_d_it.csv", index_col=0); d.columns = [int(c) for c in d.columns]

DV0 = d.values
nJ, nI = L.shape
J, I, T = range(nJ), range(nI), range(4)

CAP = {"S": 1, "M": 3, "L": 5}
FIX = {"EMS": {"S": 0.0, "M": 1.0, "L": 2.0}, "OTHER": {"S": 1.5, "M": 2.5, "L": 3.5}}

F = {(j, k): FIX["EMS" if sites.loc[j, "site_type"] == "EMS" else "OTHER"][k] for j in J for k in CAP}

is_ems = (sites["site_type"] == "EMS").values
is_fire = (sites["site_type"] == "FIREHOUSE").values
H = 6
R = np.array([1.000, 2.374, 2.942, 2.371])           # the report's R_t

def make_a(theta=8, speed=15, beta=0.15):
    kappa = 1 + beta * np.clip(R - 1, 0, None)
    miles = (L + site_dist[:, None]) / 1609.344
    tau = kappa[None, None, :] * miles[:, :, None] / speed * 60
    return tau <= theta

def reach(a, dv):                                   # share of demand that at least one site can reach
    return sum(dv[a[:, :, t].any(axis=0), t].sum() for t in T) / dv.sum()

def solve(a, dv, B=30.0, P=None, rho=0.60, s=1.5, c=0.590, target=None, time_limit=30, gap=0.001):
    pairs = [(i, j, t) for j in J for i in I for t in T if a[j, i, t] and dv[i, t] > 0]
    by_it, by_jt = defaultdict(list), defaultdict(list)
    for i, j, t in pairs:
        by_it[(i, t)].append(j); by_jt[(j, t)].append(i)
    m = gp.Model(); m.setParam("OutputFlag", 0); m.setParam("TimeLimit", time_limit); m.setParam("MIPGap", gap)
    q = m.addVars(J, list(CAP), vtype=GRB.BINARY); n = m.addVars(J, vtype=GRB.INTEGER, lb=0)
    f = m.addVars(pairs, lb=0.0)
    cost = gp.quicksum(F[j, k] * q[j, k] for j in J for k in CAP) + c * n.sum()
    m.addConstrs((q.sum(j, "*") <= 1 for j in J))
    m.addConstrs((n[j] >= q.sum(j, "*") for j in J))
    m.addConstrs((n[j] <= gp.quicksum(CAP[k] * q[j, k] for k in CAP) for j in J))
    for (i, t), js in by_it.items():
        m.addConstr(gp.quicksum(f[i, j, t] for j in js) <= dv[i, t])
    for (j, t), is_ in by_jt.items():
        m.addConstr(s * gp.quicksum(f[i, j, t] for i in is_) <= rho * H * n[j])
    if P is not None:
        m.addConstr(n.sum() <= P)
    if target is None:                                  # mode 1: best coverage within budget B
        m.addConstr(cost <= B); m.setObjective(f.sum(), GRB.MAXIMIZE)
    else:                                               # mode 2: cheapest plan reaching target coverage
        m.addConstr(f.sum() >= target * dv.sum()); m.setObjective(cost, GRB.MINIMIZE)
    m.optimize()
    if m.SolCount == 0:
        return dict(coverage=np.nan, cost=np.nan, ambulances=np.nan)
    return dict(coverage=sum(v.X for v in f.values()) / dv.sum(), cost=cost.getValue(),
                ambulances=int(round(n.sum().getValue())))

def metrics(a, dv, **kw):
    r1 = solve(a, dv, B=10, **kw)
    r2 = solve(a, dv, target=0.95, **kw)
    return dict(max_reach=round(reach(a, dv), 4), cov_at_10m=round(r1["coverage"], 4),
                cost_for_95=round(r2["cost"], 2), ambulances_for_95=r2["ambulances"])

base = metrics(make_a(), DV0)
print(base)

In [ ]:
rows = []
for theta in [6, 7, 8, 9, 10, 12]:
    rows.append(dict(theta_min=theta, **metrics(make_a(theta=theta), DV0)))
pd.DataFrame(rows)

In [ ]:
def build(a, dv, rho=0.60, s=1.5, c=0.590):
    pairs = [(i, j, t) for j in J for i in I for t in T if a[j, i, t] and dv[i, t] > 0]
    by_it, by_jt = defaultdict(list), defaultdict(list)
    for i, j, t in pairs:
        by_it[(i, t)].append(j); by_jt[(j, t)].append(i)
    m = gp.Model(); m.setParam("OutputFlag", 0)
    q = m.addVars(J, list(CAP), vtype=GRB.BINARY); n = m.addVars(J, vtype=GRB.INTEGER, lb=0)
    f = m.addVars(pairs, lb=0.0)
    cost = gp.quicksum(F[j, k] * q[j, k] for j in J for k in CAP) + c * n.sum()
    m.addConstrs((q.sum(j, "*") <= 1 for j in J))
    m.addConstrs((n[j] >= q.sum(j, "*") for j in J))
    m.addConstrs((n[j] <= gp.quicksum(CAP[k] * q[j, k] for k in CAP) for j in J))
    for (i, t), js in by_it.items():
        m.addConstr(gp.quicksum(f[i, j, t] for j in js) <= dv[i, t])
    for (j, t), is_ in by_jt.items():
        m.addConstr(s * gp.quicksum(f[i, j, t] for i in is_) <= rho * H * n[j])
    opened = {j: q.sum(j, "*") for j in J}
    for (i, j, t) in pairs:
        m.addConstr(f[i, j, t] <= dv[i, t] * opened[j])
    bud = m.addConstr(cost <= 30.0)
    m.setObjective(f.sum(), GRB.MAXIMIZE)
    return m, bud, dv.sum()

def cov_at(m, bud, total, B, time_limit=20, gap=0.001):
    bud.RHS = B
    m.setParam("TimeLimit", time_limit); m.setParam("MIPGap", gap)
    m.optimize()
    cov = m.ObjVal / total if m.SolCount else 0.0
    return cov, (m.Status == GRB.OPTIMAL)

In [ ]:
import time
a0 = make_a()
m, bud, total = build(a0, DV0)
for B in (6, 10):
    t0 = time.time()
    cov, ok = cov_at(m, bud, total, B, time_limit=30, gap=0.001)
    print(f"B={B}: coverage {cov:.4f}, proven optimal: {ok}, seconds: {time.time()-t0:.1f}", flush=True)